## Fine tuning

In [1]:
import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from langchain_community.document_loaders import UnstructuredXMLLoader
import xml.etree.ElementTree as ET
from huggingface_hub import login
from trl import SFTTrainer, SFTConfig


# Configuration for environment
os.environ["WANDB_DISABLED"] = "true"

g:\fiap\tech-challenge-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sandr\AppData\Local\Temp\ipykernel_19416\3352378986.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredXMLLoader


### Configuração geral

In [2]:
# Configuration
MODEL_ID = "epfl-llm/meditron-7B"  # Foundation model para coisas de saúde/medicina
GITHUB_REPO_ID = "abachaa/MedQuAD" # Indicado pelo professor
DATA_FILE_PATH = "data.jsonl"         # Example filename in the repo

# Hiperparametros (Otimizado para rodar em uma RTX 3060 12GB)
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "linear"
MAX_SAMPLES = 1000
MAX_SEQ_LENGTH = 512

# Login to Hugging Face
login()


### Carregando os dados do repositório git

In [3]:
import subprocess
from pathlib import Path

medquad_dir = Path("MedQuAD")

if not medquad_dir.is_dir():
    print("Pasta MedQuAD não encontrada. Clonando repositório...")
    subprocess.run(["git", "clone", "https://github.com/abachaa/MedQuAD.git"], check=True)
    print("Repositório clonado")
else:
    print("Pasta MedQuAD já existe, não é necessário clonar novamente.")

Pasta MedQuAD já existe, não é necessário clonar novamente.


In [4]:
def parse_file(file_path):
    tree = ET.parse(file_path)
    qapairs = tree.iter("QAPair")

    parsed_data = []

    for qapair in qapairs:
        question_elem = qapair.find("Question")
        answer_elem = qapair.find("Answer")

        if question_elem is not None and answer_elem is not None:
            question = question_elem.text
            answer = answer_elem.text

            if question is not None and answer is not None:
                parsed_data.append({"question": question, "answer": answer})
    return parsed_data

In [5]:
docs = []

for folder in os.listdir(medquad_dir):
    folder_path = medquad_dir / folder
    if folder_path.is_dir():
        for file in os.listdir(folder_path):
            if file.endswith(".xml"):
                file_path = folder_path / file
                docs.extend(parse_file(file_path))

df = pd.DataFrame(docs)

In [6]:
df

,question,answer
0,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points\n - Adult acute ...
1,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...
...,...,...
16402,What is (are) Parasites - Zoonotic Hookworm ?,"There are many different species of hookworms,..."
16403,Who is at risk for Parasites - Zoonotic Hookwo...,Dog and cat hookworms are found throughout the...
16404,How to diagnose Parasites - Zoonotic Hookworm ?,Cutaneous larva migrans (CLM) is a clinical di...
16405,What are the treatments for Parasites - Zoonot...,The zoonotic hookworm larvae that cause cutane...


### Baixando foundation model do Hugging Face

In [7]:
# Configuração para quantização 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir="./cache")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="cuda", cache_dir="./cache", quantization_config=bnb_config)

Loading weights: 100%|██████████| 291/291 [02:41<00:00,  1.80it/s]


### Formatando os dados para o fine tuning, usando um template alpaca

In [8]:
# Definindo o template - template inspirado no template contido no model card do Meditron
prompt_template = """###System:
You are a helpful, respectful, and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, don't share false information.

### User:
{question}

### Assistant:
{answer}"""

dataset = Dataset.from_pandas(df)

def format_alpaca_prompt(example):
    return {"text": prompt_template.format(question=example['question'], answer=example['answer'] + tokenizer.eos_token)}

dataset = dataset.map(format_alpaca_prompt, remove_columns=["question", "answer"])

dataset

Map: 100%|██████████| 16407/16407 [00:01<00:00, 12815.90 examples/s]


Dataset({
    features: ['text'],
    num_rows: 16407
})

In [9]:
model = prepare_model_for_kbit_training(model)

In [10]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Standard for many models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,746,943,488 || trainable%: 0.1243


In [12]:
training_args = SFTConfig(
    output_dir="./meditron-finetuned",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    learning_rate=LEARNING_RATE,
    num_train_epochs=3,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    logging_steps=1,
    fp16=False,
    save_strategy="steps",
    save_steps=60,
    report_to="none",
    optim="adamw_8bit",
    seed=74,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

Truncating train dataset: 100%|██████████| 16407/16407 [00:11<00:00, 1374.65 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 16407/16407 [00:07<00:00, 2204.12 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32004}.


Step,Training Loss
1,1.916049
2,1.839501
3,1.859804
4,1.839854
5,1.780910
6,1.634590
7,1.570707
8,1.544444
9,1.437534
10,1.296804


TrainOutput(global_step=3078, training_loss=0.5157073181778522, metrics={'train_runtime': 64155.4697, 'train_samples_per_second': 0.767, 'train_steps_per_second': 0.048, 'total_flos': 9.541236927534858e+17, 'train_loss': 0.5157073181778522, 'epoch': 3.0})

In [ ]:
trainer.save_model("./meditron-finetuned")

In [15]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/Kleyguerth/meditron-finetuned/commit/d7fed737341bd5b8eca51ad5499a35e2fb28d97e', commit_message='End of training', commit_description='', oid='d7fed737341bd5b8eca51ad5499a35e2fb28d97e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kleyguerth/meditron-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Kleyguerth/meditron-finetuned'), pr_revision=None, pr_num=None)